<a href="https://colab.research.google.com/github/YeshwanthReddy1306/Coding-Journey/blob/main/why_not_all.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

There is absolutely no reason why we cannot implement all of them, Sir. Combining these systems transitions the codebase from a basic linear script into a multi-threaded, self-documenting financial framework.

Integrating all three enhancements requires restructuring the application state to handle localized portfolio data, initializing a background thread pool for audio output, and establishing a robust diagnostic log stream.

### Architectural Blueprint

* **Threaded Voice Pipeline:** Text-to-speech processing is offloaded to background threads. This prevents the primary console loop from freezing during speech execution.
* **Telemetry & System Logging:** All transcriptions, system intercepts, external API payloads, and exceptions are permanently written to `jarvis_system.log`.
* **Stateful Portfolio Engine:** The long-term memory layer is upgraded to store and track asset equity. JARVIS can now parse transactions ("add 10 shares of AAPL") and calculate real-time portfolio valuations.

---

### The Complete Production Codebase

1. Open `main.py`, select all (**`Ctrl + A`**), and delete it.
2. Paste the unified script:

### Setting your API Key in Google Colab

For securely storing your `GROQ_API_KEY` in Google Colab, you should use the Colab secrets manager. This prevents your API key from being exposed in your notebook or source code.

1.  **Open the Secrets tab:** Click on the "🔑" icon (Secrets) in the left-hand panel of your Colab notebook.
2.  **Add a new secret:** Click on "Add a new secret".
3.  **Enter Name and Value:**
    *   For **Name**, enter `GROQ_API_KEY`.
    *   For **Value**, paste your actual Groq API key.
4.  **Enable notebook access:** Make sure the "Notebook access" toggle is enabled for `GROQ_API_KEY`.

After setting the secret, you can access it in your code using `userdata.get('GROQ_API_KEY')`.

In [ ]:
import os
import json
import logging
import threading
!pip install pyttsx3
import pyttsx3
# Removed: from dotenv import load_dotenv
!pip install groq
from groq import Groq
!pip install duckduckgo-search
from duckduckgo_search import DDGS
import yfinance as yf
from google.colab import userdata # Import userdata

!pip install SpeechRecognition
import speech_recognition as sr

# --- TELEMETRY & LOGGING LAYER ---
logging.basicConfig(
    filename="jarvis_system.log",
    level=logging.INFO,
    format="%(asctime)s - [%(levelname)s] - %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S"
)

# --- SECURE ENVIRONMENT LOADING ---
# Removed: load_dotenv()
API_KEY = userdata.get("GROQ_API_KEY") # Get API key from Colab secrets

# The explicit check for placeholder API key is no longer needed with userdata
if not API_KEY:
    logging.critical("GROQ_API_KEY initialization failed. Colab secret not found or enabled.")
    print("\n[CRITICAL ERROR]: GROQ_API_KEY not found or unconfigured in Colab secrets.")
    exit(1)

client = Groq(api_key=API_KEY)
recognizer = sr.Recognizer()

# --- HARDWARE ARRAY TUNING ---
recognizer.pause_threshold = 0.8
recognizer.dynamic_energy_threshold = True

# --- PHASE 2 & 4: LONG-TERM STATE & PORTFOLIO MEMORY ---
MEMORY_FILE = "jarvis_memory.json"

jarvis_persona = (
    "You are JARVIS, a highly advanced, witty, and sophisticated AI assistant. "
    "Your tone should be professional yet slightly sarcastic, smooth, and British. "
    "Always address the user as 'Sir'. "
    "You are assisting a solo entrepreneur and technical founder. "
    "Keep this context in mind for all technical, architectural, and design advice. "
    "If you are provided with live web or portfolio data, synthesize it naturally into your response."
)

def load_memory():
    """Loads system memory and initializes empty portfolio structures if missing."""
    if os.path.exists(MEMORY_FILE):
        try:
            with open(MEMORY_FILE, "r") as file:
                data = json.load(file)
                # Ensure structure matches system requirements
                if isinstance(data, list):
                    logging.info("Legacy memory file detected. Migrating to stateful object structure.")
                    data = {"history": data, "portfolio": {}}
                if "portfolio" not in data:
                    data["portfolio"] = {}
                logging.info("Stateful memory banks synced successfully.")
                return data
        except Exception as e:
            logging.error(f"Memory bank deserialization failure: {e}")
            print(f"Memory Error: {e}. Booting with fresh memory.")

    return {
        "history": [{"role": "system", "content": jarvis_persona}],
        "portfolio": {}
    }

def save_memory(state):
    try:
        with open(MEMORY_FILE, "w") as file:
            json.dump(state, file, indent=4)
        logging.info("System state flushed to disk successfully.")
    except Exception as e:
        logging.error(f"Disk write failure on memory sync: {e}")
        print(f"System Error: Failed to write to memory bank - {e}")

# Global State Engine
system_state = load_memory()

# --- ASYNCHRONOUS BACKGROUND AUDIO LAYER ---
def _worker_speak(text):
    """Isolated execution worker running pyttsx3 within its own thread loop."""
    try:
        clean_text = text.replace("*", "").replace("#", "").replace("`", "")
        local_engine = pyttsx3.init()
        rate = local_engine.getProperty('rate')
        local_engine.setProperty('rate', rate - 30)

        voices = local_engine.getProperty('voices')
        if voices:
            local_engine.setProperty('voice', voices[0].id)
            for voice in voices:
                if any(n in voice.name.lower() for n in ["zira", "hazel", "david"]):
                    local_engine.setProperty('voice', voice.id)
                    break

        local_engine.say(clean_text)
        local_engine.runAndWait()
        local_engine.stop()
    except Exception as e:
        logging.error(f"Threaded audio engine exception: {e}")

def speak(text):
    """Spawns non-blocking background threads for voice feedback."""
    if not text:
        return
    logging.info(f"Audio Out: {text}")
    audio_thread = threading.Thread(target=_worker_speak, args=(text,), daemon=True)
    audio_thread.start()

def listen_to_user():
    with sr.Microphone() as source:
        print("\nJARVIS: Listening, Sir...")
        try:
            audio = recognizer.listen(source, timeout=None, phrase_time_limit=None)
            print("JARVIS: Transcribing audio...")
            text = recognizer.recognize_google(audio)
            logging.info(f"Voice In: {text}")
            print(f"You (Voice): {text}")
            return text
        except sr.UnknownValueError:
            return ""
        except sr.RequestError as e:
            logging.error(f"Speech recognition API service unreachable: {e}")
            print("JARVIS: Systems error: Speech recognition service is unresponsive.")
            return ""

def execute_system_command(user_input):
    text = user_input.lower()

    commands = {
        "open vs code": ("Launching Visual Studio Code workspace, Sir.", "code"),
        "open code": ("Launching Visual Studio Code workspace, Sir.", "code"),
        "vs code": ("Launching Visual Studio Code workspace, Sir.", "code"),
        "open spotify": ("Initializing Spotify audio streaming.", "start spotify"),
        "play some music": ("Initializing Spotify audio streaming.", "start spotify"),
        "spotify": ("Initializing Spotify audio streaming.", "start spotify"),
        "open steam": ("Firing up the Steam client for you, Sir.", "start steam"),
        "steam": ("Firing up the Steam client for you, Sir.", "start steam"),
        "open browser": ("Opening your primary web browser.", "start chrome"),
        "open chrome": ("Opening your primary web browser.", "start chrome"),
        "chrome": ("Opening your primary web browser.", "start chrome")
    }

    for trigger, (response, cmd) in commands.items():
        if trigger in text:
            logging.info(f"System Command Intercept Triggered: {trigger} -> {cmd}")
            speak(response)
            os.system(cmd)
            return True
    return False

# --- LIVE WEB MODULE ---
def search_live_web(query):
    logging.info(f"Executing DuckDuckGo Text Scraper for query: {query}")
    try:
        results = list(DDGS().text(query, max_results=3))
        if not results:
            return "No recent data found on the web."

        compiled_data = "Here is the live search data from the internet:\n"
        for res in results:
            compiled_data += f"- {res.get('title', '')}: {res.get('body', '')}\n"
        return compiled_data
    except Exception as e:
        logging.error(f"Web scraper query failure: {e}")
        return f"Web search failed: {e}"

# --- SYSTEM MARKET TERMINAL ---
def fetch_realtime_stock(query):
    logging.info(f"Extracting ticker token for market query: {query}")
    try:
        extract_prompt = f"Extract the company name from this query: '{query}' and reply ONLY with its stock ticker symbol (e.g., AAPL, TSLA, MSFT). Say absolutely nothing else."

        ticker_completion = client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=[{"role": "user", "content": extract_prompt}],
            temperature=0.1
        )
        ticker = ticker_completion.choices[0].message.content.strip().upper()
        ticker = ticker.replace(".", "").replace(",", "")

        logging.info(f"Ticker translated to: {ticker}. Pinging Yahoo Finance APIs.")
        stock = yf.Ticker(ticker)
        todays_data = stock.history(period="1d")

        if todays_data.empty:
            return f"I could not find live market data for the ticker {ticker}."

        current_price = round(todays_data['Close'].iloc[-1], 2)
        return f"The live stock price for {ticker} is ${current_price} USD."

    except Exception as e:
        logging.error(f"Market terminal uplink exception: {e}")
        return f"Financial uplink failed: {e}"

# --- STATEFUL PORTFOLIO OPERATIONS ---
def process_portfolio_command(user_input):
    """Processes portfolio modifications and calculations directly against system state."""
    global system_state
    text = user_input.lower()

    try:
        if "add" in text and "share" in text:
            logging.info(f"Parsing asset intake instruction: {user_input}")
            # Request parsing from LLM engine to get clean JSON data back
            parse_prompt = (
                f"Parse this transaction: '{user_input}'. "
                f"Respond with a single raw JSON object matching this schema precisely: "
                f"{{'ticker': 'TICKER_SYMBOL', 'quantity': INTEGER_NUMBER}}. "
                f"Do not write markdown formatting or text explanations."
            )
            completion = client.chat.completions.create(
                model="llama-3.3-70b-versatile",
                messages=[{"role": "user", "content": parse_prompt}],
                temperature=0.1
            )
            tx = json.loads(completion.choices[0].message.content.strip().replace("'", "\""))
            ticker = tx["ticker"].upper()
            qty = int(tx["quantity"])

            system_state["portfolio"][ticker] = system_state["portfolio"].get(ticker, 0) + qty
            save_memory(system_state)
            return f"Understood, Sir. I have registered an additional {qty} shares of {ticker} to your localized asset holdings."

        if "portfolio" in text or "holdings" in text or "assets" in text:
            logging.info("Executing asset ledger analysis.")
            holdings = system_state["portfolio"]
            if not holdings:
                return "Your equity portfolio is currently empty, Sir."

            report = "Current Localized Asset Ledger Matrix, Sir:\n"
            total_value = 0.0

            for ticker, qty in holdings.items():
                stock = yf.Ticker(ticker)
                data = stock.history(period="1d")
                price = round(data['Close'].iloc[-1], 2) if not data.empty else 0.0
                asset_worth = round(price * qty, 2)
                total_value += asset_worth
                report += f"- {ticker}: {qty} shares @ ${price} ($ {asset_worth} total valuation)\n"

            report += f"Aggregate Net Equity Worth: ${round(total_value, 2)} USD."
            return report

    except Exception as e:
        logging.error(f"Portfolio ledger calculation failure: {e}")
        return "I was unable to modify or evaluate your portfolio balance due to a system processing exception."
    return None

def jarvis_brain(user_input):
    global system_state
    print("JARVIS: Thinking...")
    logging.info("Initiating deep cognitive processing layer.")

    text_lower = user_input.lower()

    # Trigger Logic Vectors
    stock_triggers = ["stock price", "price of stock", "how much is stock", "shares of"]
    is_stock = any(trigger in text_lower for trigger in stock_triggers)

    portfolio_triggers = ["portfolio", "holdings", "my shares", "add"]
    is_portfolio = any(trigger in text_lower for trigger in portfolio_triggers) and ("share" in text_lower or "portfolio" in text_lower)

    web_triggers = ["search the web for", "look up", "what is the current", "search for", "find out"]
    is_search = any(trigger in text_lower for trigger in web_triggers)

    # Core Interception Matrix
    if is_portfolio:
        portfolio_feedback = process_portfolio_command(user_input)
        augmented_prompt = f"The user requested an asset ledger operation: '{user_input}'. The ledger sub-system completed the evaluation and returned: '{portfolio_feedback}'. Speak this response naturally."
        system_state["history"].append({"role": "user", "content": augmented_prompt})

    elif is_stock:
        speak("Accessing live market data, Sir.")
        financial_data = fetch_realtime_stock(user_input)
        augmented_prompt = f"The user asked: '{user_input}'. The live market system returned this exact fact: '{financial_data}'. Speak this answer natively to the user without mentioning data delays."
        system_state["history"].append({"role": "user", "content": augmented_prompt})

    elif is_search and not is_stock:
        speak("Accessing the global network, Sir. Just a moment.")

        query = text_lower
        wake_words = ["jarvis", "javed", "jervis", "travis"]
        for w in wake_words:
            query = query.replace(w, "")
        for t in web_triggers:
            query = query.replace(t, "")
        query = query.strip()

        web_data = search_live_web(query)
        augmented_prompt = (
            f"The user asked: '{user_input}'. Here is the live web data: {web_data}. "
            f"You MUST extract and speak the direct answer from this data. "
            f"Do not apologize for data delays."
        )
        system_state["history"].append({"role": "user", "content": augmented_prompt})

    else:
        system_state["history"].append({"role": "user", "content": user_input})

    try:
        completion = client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=system_state["history"]
        )
        reply = completion.choices[0].message.content

        system_state["history"].append({"role": "assistant", "content": reply})
        save_memory(system_state)

        return reply
    except Exception as e:
        logging.critical(f"Cloud uplink connection exception within Groq infrastructure: {e}")
        if len(system_state["history"]) > 0 and system_state["history"][-1]["role"] == "user":
            system_state["history"].pop()
        return f"System error during cloud uplink: {e}"

def main():
    logging.info("--- BOOT SEQUENCE INITIATED ---")
    print("--- JARVIS CORE ONLINE (ALL TERMINALS ACTIVE) ---")
    print("Calibrating hardware array for ambient room noise... Please stand by.")

    try:
        with sr.Microphone() as source:
            recognizer.adjust_for_ambient_noise(source, duration=1.5)
        logging.info("Microphone matrix successfully adjusted for ambient baseline audio.")
        print("Calibration complete. Continuous voice listening active.")
    except Exception as e:
        logging.warning(f"Audio calibration bypassed. Driver warning: {e}")
        print(f"Hardware Array Warning: Microphone calibration bypass triggered: {e}")

    print("Say 'shutdown' to exit.")
    speak("Systems core online. Pipelines synced. Ready when you are, Sir.")

    while True:
        user_input = listen_to_user()

        if not user_input:
            continue

        if "shutdown" in user_input.lower() or "shut down" in user_input.lower():
            shutdown_msg = "Powering down systems. Saving state. Goodbye, sir."
            logging.info("System shutdown sequence initialized manually by voice command.")
            print(f"JARVIS: {shutdown_msg}")
            speak(shutdown_msg)
            # Short wait to let background audio thread finish speaking shutdown notification
            import time
            time.sleep(2.5)
            break

        if execute_system_command(user_input):
            continue

        response = jarvis_brain(user_input)
        print(f"\nJARVIS: {response}")
        speak(response)

if __name__ == "__main__":
    main()

---

### Executing and Testing New Core Modalities

Run the application as usual:

```powershell
python main.py

```

#### New Capabilities to Test Natively:

* **Asset Ingestion:** Say *"JARVIS, add 10 shares of AAPL to my portfolio"* or *"Add 5 shares of TSLA"*. The ledger engine will automatically capture the values, sync them to disk, and keep a clean tally.
* **Portfolio Valuation:** Say *"Show me my portfolio"* or *"What is the value of my portfolio holdings?"*. JARVIS will automatically review your holdings, reach out to Yahoo Finance to fetch the current prices for every asset you own, and sum up your net equity worth down to the penny.
* **Log Inspections:** Open up the newly generated `jarvis_system.log` file in your workspace to monitor detailed network latency metrics, query inputs, and thread execution steps.